# Hyperparameter Tuning: Practical Search Budgets
# 超参数调优：实用搜索预算

Scenario: a forecasting platform must tune within strict time budgets. PipelineTS supports explicit `PipelineConfigs`, double-underscore model kwargs, and SmartRouter HPO strategies.

场景：预测平台需要在严格时间预算内调参。PipelineTS 支持显式 `PipelineConfigs`、双下划线模型参数和 SmartRouter HPO 策略。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
from PipelineTS.pipeline import ModelPipeline, PipelineConfigs, SmartRouter

data = make_retail_demand(n_days=220, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-21].copy(), data.iloc[-21:].copy()

In [ ]:
pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    include_models=["random_forest", "extra_forest"],
    quantile=None,
    cv=2,
    random_forest__n_estimators=100,
    random_forest__max_depth=8,
    extra_forest__n_estimators=160,
    extra_forest__max_depth=12,
)
pipe.fit(train, valid_data=valid)
pipe.leader_board_

In [ ]:
configs = PipelineConfigs([
    ("random_forest", "rf_80trees_lag7", {
        "init_configs": {"n_estimators": 80, "max_depth": 8, "random_state": 42},
        "pipeline_configs": {"lags": 7, "scaler": None},
    }),
    ("random_forest", "rf_160trees_lag21", {
        "init_configs": {"n_estimators": 160, "max_depth": 12, "random_state": 42},
        "pipeline_configs": {"lags": 21, "scaler": True},
    }),
])

tuned_pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    include_models=["random_forest"],
    configs=configs,
    quantile=None,
    cv=2,
)
tuned_pipe.fit(train, valid_data=valid)
tuned_pipe.leader_board_

In [ ]:
router = SmartRouter(
    time_col="date",
    target_col="sales",
    preset="medium_quality",
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    search_strategy="auto",
    hpo_strategy="auto",
    hpo_n_trials=5,
    hpo_timeout_per_model=20,
    time_limit=120,
)
router.fit(train, valid_data=valid)

print("Active HPO strategy:", router._active_hpo_strategy_)
print("HPO results:", router._hpo_results)
router.leader_board_

In [ ]:
print("Strategy keys:", router.strategy_.keys())
print("Autonomy summary:")
router.autonomy_summary_